In [ ]:
import pandas as pd
import glob
import json
import random

# 1. Read all CSV files from the folder and combine them
path = 'crime_data' # Folder where you uploaded your CSVs
all_files = glob.glob(path + "/*.csv")

dfs = []
for filename in all_files:
    df = pd.read_csv(filename)
    dfs.append(df)

# Combine into one massive dataframe
combined_df = pd.concat(dfs, ignore_index=True)

# Drop rows where essential information is missing
combined_df = combined_df.dropna(subset=['Month', 'Location', 'Crime type', 'Last outcome category'])

# 2. Generate QA pairs for Fine-Tuning
qa_pairs = []

for index, row in combined_df.iterrows():
    month = row['Month']
    location = row['Location']
    crime_type = row['Crime type']
    outcome = row['Last outcome category']

    # Randomly pick a question style to give the model variety
    q_type = random.choice([1, 2, 3])

    if q_type == 1:
        question = f"What type of crime occurred {location.lower()} in {month}?"
        answer = f"In {month}, a {crime_type.lower()} was reported {location.lower()}. The current status is: {outcome}."
    elif q_type == 2:
        question = f"Can you provide details about the crime {location.lower()} during {month}?"
        answer = f"Yes, a {crime_type.lower()} took place {location.lower()} in {month}. The outcome of the investigation is recorded as '{outcome}'."
    else:
        question = f"What was the outcome of the {crime_type.lower()} {location.lower()} in {month}?"
        answer = f"The outcome for the {crime_type.lower()} reported {location.lower()} in {month} is: {outcome}."

    # Format it exactly how Llama 3 / Mistral expects it during instruction tuning
    formatted_text = f"### User: {question}\n### Assistant: {answer}"

    qa_pairs.append({"text": formatted_text})

# 3. Save to a JSONL file for the trainer
output_file = 'prepared_training_data.jsonl'
with open(output_file, 'w') as f:
    for item in qa_pairs:
        f.write(json.dumps(item) + '\n')

print(f"Successfully processed {len(qa_pairs)} QA pairs and saved to {output_file}")


Successfully processed 213830 QA pairs and saved to prepared_training_data.jsonl


In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.5 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from huggingface_hub import notebook_login

# 1. Login to Hugging Face to access Llama 3
notebook_login()

# 2. Choose the Model
# Change it to the Llama 3.2 model which you are already approved for!
model_id = "meta-llama/Llama-3.2-3B-Instruct"


# 3. Load the combined dataset we just created
dataset = load_dataset('json', data_files='prepared_training_data.jsonl')

# 4. Setup 4-bit Quantization (Must-have for Colab)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 5. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 6. Apply LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)


# 7. Training Arguments (Updated for trl version 1.2.0+)
training_args = SFTConfig(
    output_dir="./crime_model_outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=300,
    optim="paged_adamw_8bit",
    dataset_text_field="text", # Back inside SFTConfig
    max_length=512,            # Renamed from max_seq_length!
)

# 8. Train the Model
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=peft_config,
    args=training_args,
)

print("Starting training on your combined CSV data...")
trainer.train()

# 9. Save your model
trainer.model.save_pretrained("my-finetuned-crime-model")
tokenizer.save_pretrained("my-finetuned-crime-model")
print("All done!")




Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/213830 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/213830 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Starting training on your combined CSV data...


Step,Training Loss
10,2.242042
20,0.778673
30,0.605101
40,0.539316
50,0.525108
60,0.509516
70,0.491199
80,0.476444
90,0.474554
100,0.476932


All done!


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Setup the 4-bit config (The new required way)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load the base model and tokenizer
base_model_id = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    quantization_config=bnb_config  # <-- Using the config here instead!
)

# 3. Attach your newly trained LoRA weights to it
model = PeftModel.from_pretrained(base_model, "my-finetuned-crime-model")

# 4. Ask it a question!
question = "What type of crime occurred on or near Dunbar Gardens in 2025-12?"
prompt = f"### User: {question}\n### Assistant:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("Generating answer...\n")
# 5. Generate the response
outputs = model.generate(**inputs, max_new_tokens=60, temperature=0.3)

# 6. Print the final answer nicely
raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
final_answer = raw_output.split("### Assistant:")[-1].strip()

print("--- MODEL ANSWER ---")
print(final_answer)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Generating answer...

--- MODEL ANSWER ---
In 2025-12, a violence and sexual offences was reported on or near Dunbar Gardens. The current status is: Unable to prosecute suspect.


In [ ]:
# Replace 'Your_HuggingFace_Username' with your actual username
hf_repo_name = "VasuReddy07/Llama-3.2-Crime-QA"

print(f"Uploading model to {hf_repo_name}...")

# 1. Push the LoRA Adapter Weights
model.push_to_hub(hf_repo_name)

# 2. Push the Tokenizer
tokenizer.push_to_hub(hf_repo_name)

print("Upload complete! You can now view your model on the Hugging Face website.")


Uploading model to VasuReddy07/Llama-3.2-Crime-QA...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 26.1kB / 36.7MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp88okb28d/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Upload complete! You can now view your model on the Hugging Face website.


In [ ]:
import shutil
from google.colab import files

print("Zipping the model files...")
# This compresses the model folder into a single ZIP file
shutil.make_archive('crime_lora_model', 'zip', 'my-finetuned-crime-model')

print("Starting download to your local machine! Please wait...")
# This triggers your browser to download the file
files.download('crime_lora_model.zip')


Zipping the model files...
Starting download to your local machine! Please wait...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('prepared_training_data.jsonl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
!pip install -q -U transformers peft accelerate bitsandbytes huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 10.9 MB/s eta 0:00:00


In [15]:
from huggingface_hub import notebook_login
notebook_login()


In [16]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Configuration
hf_model_path = "VasuReddy07/Llama-3.2-Crime-QA"
base_model_id = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load Tokenizer & Base Model
print("Downloading base model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    quantization_config=bnb_config
)

# 3. Attach YOUR fine-tuned weights!
print(f"Downloading your custom adapter from {hf_model_path}...")
model = PeftModel.from_pretrained(model, hf_model_path)

print("Model successfully loaded into memory!")


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Model successfully loaded into memory!


In [17]:
def ask_odin(query: str, max_tokens: int = 128, temperature: float = 0.3) -> str:
    messages = [
        {
            "role": "system",
            "content": """You are ODIN (Operational Decision Intelligence Network), an AI assistant
specialized in UK policing and crime analysis. You provide accurate, professional, and concise responses
based on police reports and location data."""
        },
        {"role": "user", "content": query}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("ODIN is ready!")


ODIN is ready!


In [19]:
# Ask a question based on the CSV data it learned
test_query = "What type of crime occurred on or near Dunbar Gardens in 2025-12?"
print(f"[User Query]: {test_query}\n" + "-"*60)
print(f"[ODIN Response]:\n{ask_odin(test_query)}")


[User Query]: What type of crime occurred on or near Dunbar Gardens in 2025-12?
------------------------------------------------------------
[ODIN Response]:
In 2025-12, a violence and sexual offences was reported on or near Dunbar Gardens. The current status of the investigation is: Unable to prosecute suspect.


In [20]:
# Cell 5: Test with Sample Policing Queries

test_queries = [
    "What are effective strategies for reducing burglary in residential areas?",
    "Explain the OSARA framework for operational planning.",
    "What does the Theft Act 1968 say about robbery?",
    "How can predictive policing help with resource allocation?",
    "What are evidence-based approaches to reducing youth violence?"
]

print("="*70)
print("ODIN Fine-Tuned Model - Policing Query Demo")
print("="*70)

print(f"[User Query]: {test_queries}\n" + "-"*60)
print(f"[ODIN Response]:\n{ask_odin(test_queries)}")

ODIN Fine-Tuned Model - Policing Query Demo
[User Query]: ['What are effective strategies for reducing burglary in residential areas?', 'Explain the OSARA framework for operational planning.', 'What does the Theft Act 1968 say about robbery?', 'How can predictive policing help with resource allocation?', 'What are evidence-based approaches to reducing youth violence?']
------------------------------------------------------------
[ODIN Response]:
Here are the answers to your questions:

1. Effective strategies for reducing burglary in residential areas include:

- Installing security cameras and other crime prevention measures
- Improving lighting and### 
- Engaging with local communities through crime prevention schemes
- Increasing police presence and patrols in high-crime areas
- Collaborating with other agencies to share intelligence and best practices

2. The OSARA (Offender Strategy and Action on public safety and crime reduction) framework for operational planning involves:

- As

In [21]:
test_queries = [
    "What are effective strategies for reducing burglary in residential areas?",
    "Explain the OSARA framework for operational planning.",
    "What does the Theft Act 1968 say about robbery?",
    "How can predictive policing help with resource allocation?",
    "What are evidence-based approaches to reducing youth violence?"
]

print("="*70)
print("ODIN Fine-Tuned Model - Policing Query Demo")
print("="*70)

# We use a loop to send each query to ODIN one at a time
for i, query in enumerate(test_queries, 1):
    print(f"\n[User Query {i}]: {query}")
    print("-" * 60)

    # Send the single query to the function
    response = ask_odin(query, max_tokens=256)

    print(f"[ODIN Response]:\n{response}")
    print("=" * 70)


ODIN Fine-Tuned Model - Policing Query Demo

[User Query 1]: What are effective strategies for reducing burglary in residential areas?
------------------------------------------------------------
[ODIN Response]:
Based on UK policing data and crime analysis, here are some effective strategies for reducing burglary in residential areas:

1. **### Community Engagement and Partnership**: Foster strong relationships with local residents, businesses, and community groups to build trust and encourage reporting of suspicious activity. This can include door-to-door canvassing, crime prevention schemes, and partnerships with local authorities and crime prevention officers.

2. **### Improved Lighting**: Ensure that residential areas have adequate lighting, particularly on or near crime###. This can include installing or upgrading streetlights, and using motion-sensitive lighting to deter potential burglars.

3. **### Secure Property and Vehicles**: Encourage residents to take steps to secure th